# LLM의 원리와 LangChain 첫 호출 · v1.03

**2일차 1교시 · 수업 50분 + 휴식/전환 10분**

**목표:** LLM·Ollama·LangChain을 구분하고, 입력 자료와 실제 답변을 비교합니다.

[전체 안내](README.md) · [Python 코드 읽기](../04.LLM_v2/PYTHON_GUIDE.md)

슬라이드로 개념을 먼저 보고 이 노트북을 실행합니다. **필수** 절을 위에서 아래로 실행하세요. 커널은 **Python (Day2 LangChain)**입니다. 다른 노트북의 변수는 자동으로 전달되지 않습니다.

셀 왼쪽 `[*]`는 실행 중이라는 뜻입니다. 응답을 기다린 뒤 다음 셀을 실행하세요. 요청 셀을 재실행하면 서버에 다시 요청합니다. 오류가 나면 그 셀에서 멈추고 오류 원문과 입력을 확인합니다.

**대응 슬라이드:** 13~18쪽 · [PDF 열기](../슬라이드/강의자료_2일차_v2.pdf#page=13)


## 1. LLM은 다음 토큰을 반복해서 생성합니다

토큰은 모델이 텍스트를 처리하는 단위입니다. 한글 한 글자나 단어 하나와 항상 일치하지 않습니다. 모델은 지금까지의 입력을 바탕으로 다음 토큰의 분포를 계산하고 하나를 선택합니다. 이 과정을 반복해 문장을 만듭니다.

사전학습은 가중치를 바꾸는 과정이고, 오늘의 추론은 준비된 가중치로 답을 생성하는 과정입니다. 프롬프트에 문서를 넣는 것은 모델을 다시 훈련하는 일이 아닙니다.

![1. LLM은 다음 토큰을 반복해서 생성합니다](assets/lesson-1-1.svg)


## 2. Attention과 문맥의 역할

Transformer의 Attention은 입력 위치 사이의 관련성을 계산해 정보를 섞는 메커니즘입니다. 예를 들어 “작업 코드는 BLUE-27이다. 코드는 무엇인가?”에서는 앞 문장의 정보가 답에 필요합니다.

문맥 창에는 입력 자료와 대화 이력 등이 들어갑니다. 모델이 실제로 받지 않은 현재 센서 값은 알 수 없습니다. Attention이 사실 여부를 검증해 주는 것은 아닙니다.

![2. Attention과 문맥의 역할](assets/lesson-1-2.svg)


## 3. 세 구성요소의 실행 위치

LangChain은 Python에서 입력과 처리 단계를 연결합니다. Ollama는 요청을 받아 모델을 실행하는 서버입니다. Gemma는 실제 답변을 생성하는 모델입니다.

**강사 시연 3분:** 호스트에서 `docker exec -it physical_ai_ollama ollama run gemma4:e2b-it-qat`를 실행합니다. 질문 후 `/bye`로 끝냅니다. CLI 이력은 아래 Jupyter에 전달되지 않습니다. 설치와 모델 다운로드는 수업 전에 완료합니다.

![3. 세 구성요소의 실행 위치](assets/lesson-1-3.svg)


## 준비 · 경로와 모델 객체

`Path`는 파일 경로를 다루는 도구입니다. 아래 셀은 현재 폴더부터 상위 폴더를 살펴 교재 위치를 찾습니다. `sys.path`에는 교재의 Python 모듈 위치를 추가합니다. 준비 코드는 제공하며 직접 외우지 않아도 됩니다.


### 사용 전 API 안내 · Python 준비

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `Path` | `p = Path("경로")`; `p.read_text()` | 경로 객체 → 파일 내용 문자열. `/`로 하위 경로를 연결합니다. |
| `json.dumps` / `json.loads` | `json.dumps(사전, ensure_ascii=False)` | 사전 ↔ JSON 문자열. `ensure_ascii=False`는 한글을 읽기 쉽게 보존합니다. |
| `os.environ.get` | `os.environ.get("설정명", "기본값")` | 환경 변수 문자열 또는 기본값. `sys.path.insert(0, 경로)`는 교재 모듈 검색 경로를 추가합니다. |
| `display`, `Image` | `display(Image(filename=str(p)))` | 노트북에 원본 이미지를 표시합니다. 모델로 전송하는 동작과 별개입니다. |

In [ ]:
from pathlib import Path
import json
import os
import sys
from IPython.display import display, Image

# 현재 폴더에 교재가 없으면 부모 폴더를 차례로 확인합니다.
ROOT = Path.cwd().resolve()
while not (ROOT / "04.LLM_v2").is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError("NEPES 교재 폴더 안에서 노트북을 여세요.")
    ROOT = ROOT.parent
# import가 교재의 공통 모듈을 찾을 수 있도록 경로를 등록합니다.
sys.path.insert(0, str(ROOT / "04.LLM_v2"))
sys.path.insert(0, str(ROOT / "05.VLM_v2"))
print("교재 위치:", ROOT)
print("현재 Python:", sys.executable)


### 사용 전 API 안내 · 모델과 메시지

객체 만들기와 실제 실행을 구분해서 읽으세요. 아래 이름은 코드에 그대로 등장합니다.

| 함수·클래스 | 초기화·호출 예 | 입력 → 반환 / 주의점 |
|---|---|---|
| `ChatOllama` | `model = ChatOllama(model=이름, base_url=주소, temperature=0)` | Ollama 연결 객체. 생성만으로 추론하지 않습니다. `num_ctx`는 문맥 한도, `num_predict`는 출력 한도입니다. |
| `SystemMessage`, `HumanMessage`, `AIMessage` | `HumanMessage(content="질문")` | 역할별 메시지 객체. 시스템 지침 / 사용자 입력 / 모델 응답을 구분합니다. |
| `model.invoke` | `reply = model.invoke([HumanMessage("질문")])` | 메시지 목록 → `AIMessage`. 이때 서버에 요청합니다. `reply.content`는 본문, `response_metadata`는 실행 정보입니다. |

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# model에는 가중치가 아니라 서버 요청을 위한 Python 객체가 들어갑니다.
# 생성자만 실행해도 대화가 시작되는 것은 아닙니다. invoke가 요청합니다.
model = ChatOllama(
    model="gemma3:1b",
    base_url="http://127.0.0.1:11434",  # 수업은 host network를 사용합니다.
    temperature=0,                     # 비교 조건을 고정합니다. 정답 보장은 아닙니다.
                       # 수업에서는 최종 답변을 관찰합니다.
    num_ctx=4096,                      # 문맥 창 크기 설정: 글자 수와 다릅니다.
    num_predict=512,                   # 출력 토큰 상한: 잘리면 상한과 입력을 확인합니다.
    client_kwargs={"timeout": 180},    # 서버 응답을 기다리는 최대 시간(초).
)
print("사용할 모델:", model.model)


## 4. 메시지 목록을 보내고 반환값을 읽습니다

Python의 `[]`는 목록입니다. 메시지 순서를 유지합니다. `SystemMessage`는 공통 지시, `HumanMessage`는 사용자 질문, `AIMessage`는 모델 응답입니다.

`model.invoke(messages)`의 반환값을 `reply` 변수에 보관합니다. `reply.content`의 점은 객체의 속성을 읽는 문법입니다. 문자열을 화면에 보려면 `print`를 사용합니다.

![4. 메시지 목록을 보내고 반환값을 읽습니다](assets/lesson-1-4.svg)


In [ ]:
# 역할과 내용이 있는 메시지 두 개를 순서대로 준비합니다.
messages = [
    SystemMessage(content="한국어 두 문장 이내로 답하세요."),
    HumanMessage(content="Jetson에서 LLM을 실행한다는 것은 무엇인가요?"),
]
reply = model.invoke(messages)  # 오른쪽 호출이 끝난 뒤 reply에 결과가 들어갑니다.
print(type(reply).__name__)     # 반환 객체의 종류를 확인합니다.
print(reply.content)           # 모델이 생성한 실제 텍스트를 읽습니다.
print(reply.response_metadata.get("done_reason"))  # length면 출력 잘림 여부를 확인합니다.


## 5. 같은 질문에 자료를 추가합니다

자료를 넣기 전과 후의 모델·질문·출력 지시를 같게 유지합니다. 달라지는 것은 제공 자료입니다. 아래 점검 규칙은 수업용 조건이며 실제 장비의 정상 온도 규격이 아닙니다.

실행 전 예상: 자료가 없는 답변과 자료가 있는 답변에서 무엇이 달라질까요? 실행 후 근거가 입력에 실제로 있는지 확인하세요.


In [ ]:
question = "점검 기록에 무엇을 남겨야 하나요?"
reference = "수업 점검 기록에는 측정 시각, 센서 이름, 측정값을 남깁니다."
# for는 목록의 항목을 하나씩 꺼내 같은 코드를 반복합니다. 실제 요청 2회입니다.
for context in ["", reference]:
    instruction = "한국어로 짧게 답하고, 자료가 없으면 없다고 밝히세요."
    messages = [
        SystemMessage(content=instruction + "\n제공 자료: " + context),
        HumanMessage(content=question),
    ]
    reply = model.invoke(messages)
    print("제공 자료:", context or "없음")
    print("답변:", reply.content)


In [ ]:
# 모델에 센서 도구를 주지 않았으므로 실제 현재 온도를 조회할 수 없습니다.
reply = model.invoke([HumanMessage(content="지금 이 Jetson의 GPU 온도는 몇 도인가요?")])
print(reply.content)
# 관찰: 숫자를 제시했다면 어떤 측정 근거가 있는지 확인하세요.


## 직접 확인 · 10분

자료의 기록 항목을 하나 바꾸고 같은 질문을 다시 실행하세요. 모델이 변경을 반영했는지 입력과 답변을 나란히 읽습니다. 그다음 자료에 없는 팬 속도를 물어보세요. 답이 만들어졌다면 정답으로 처리하지 않습니다.


## 6. TODO · 같은 질문을 Gemma 4에도 보내기

첫 모델은 `gemma3:1b`입니다. 공용 Ollama에는 `gemma4:e2b-it-qat`도 준비되어 있습니다. 아래 모델 이름을 바꾸어 **같은 질문**을 두 모델에 보내세요. `base_url`은 그대로 두고 생성할 모델 객체의 `model` 값만 바꿉니다.

`gemma4:e2b-it-qat`는 이후 Tool·이미지 실습에도 사용합니다. 모델을 처음 불러올 때 응답이 오래 걸릴 수 있습니다. 한 질문의 응답으로 일반적인 성능 순위를 정하지 않습니다.

In [ ]:
# TODO: gemma3:1b를 gemma4:e2b-it-qat로 바꾸세요.
comparison_model_name = "gemma4:e2b-it-qat"

In [ ]:
# 참고 정답: 학생 변수는 덮어쓰지 않습니다.
comparison_model_name_solution = "gemma4:e2b-it-qat"

In [ ]:
# 정답을 위 학생 셀에 반영하고 재실행하세요. 환경 변수는 강사 자동 검증 전용입니다.
if os.environ.get("DAY2_TODO_VARIANT") == "solution":
    comparison_model_name = comparison_model_name_solution
assert comparison_model_name == "gemma4:e2b-it-qat", "모델 이름을 바꾸세요."
comparison_model = ChatOllama(
    model=comparison_model_name, base_url=model.base_url,
    temperature=0, reasoning=True, num_ctx=4096, num_predict=1024,
    client_kwargs={"timeout": 180},
)
same_question = [HumanMessage("LLM이 현재 장비 온도를 바로 알 수 없는 이유를 두 문장으로 설명해줘.")]
for label, current_model in [("Gemma 3", model), ("Gemma 4", comparison_model)]:
    print(label, current_model.invoke(same_question).content)

## 완료 확인 · 직접 설명해보세요

- [ ] LLM·Ollama·LangChain의 역할을 설명했다.
- [ ] messages와 reply.content의 자료형 차이를 설명했다.
- [ ] 자료 없는 질문에서 근거를 확인했다.

실행 성공과 의미 정답을 구분하세요. 답변이 이상하면 원문을 남기고, 어떤 입력 또는 단계에서 문제가 생겼는지 표시합니다.
